# CS412 Week 7 — Facial Emotion Recognition (FER2013)
Name: Jhey Gulde

Course and Section: CS412 - CS4A

Roles: Individual work. Jhey Gulde did the data checks, face detection, feature extraction, model training, evaluation and write-up.

Pipeline: **FER2013 folders → Sanity check → Face detection (Haar) → Face crop → MobileNetV2 embeddings (Track B) → Logistic Regression & Random Forest → Evaluation → `predict_emotion()` on unseen images**

## STEP 0 — Environment and configuration

In [ ]:
# All packages are already installed on this machine. On a fresh machine, uncomment:
# %pip install -q opencv-python numpy pandas scikit-learn matplotlib torch torchvision

In [ ]:
from pathlib import Path
import hashlib
import random
import shutil
import time
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
TRAIN_DIR, TEST_DIR = DATA_DIR / "train", DATA_DIR / "test"
MODELS_DIR = ROOT / "models"
HAAR_PATH = MODELS_DIR / "haarcascade_frontalface_default.xml"
FEAT_DIR = ROOT / "features"
OUT_DIR = ROOT / "OUTPUTS"
UNSEEN_DIR = ROOT / "unseen_images"
for d in (FEAT_DIR, OUT_DIR, UNSEEN_DIR):
    d.mkdir(exist_ok=True)

CLASSES = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.hub.set_dir(str(MODELS_DIR / "torch_hub"))   # MobileNetV2 weights are cached here

for name, mod in [("OpenCV", cv2), ("NumPy", np), ("pandas", pd), ("scikit-learn", sklearn),
                  ("PyTorch", torch), ("torchvision", torchvision)]:
    print(f"{name:<13}{mod.__version__}")
print("Device       ", "cuda" if torch.cuda.is_available() else "cpu")

## STEP 1 — Sanity check: class folders, image counts, image spec
FER2013 (Kaggle `msambare/fer2013` release) ships with a fixed split: `data/train` is the original *Training* partition and `data/test` is *PublicTest + PrivateTest*. We keep that split as given, so the test images are never touched during training or tuning.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}

for split_dir in (TRAIN_DIR, TEST_DIR):
    missing = [c for c in CLASSES if not (split_dir / c).is_dir()]
    assert not missing, f"{split_dir} is missing class folders: {missing}"
print("OK: all 7 class folders exist in data/train and data/test ->", CLASSES)

def list_images(split_dir):
    rows = [{"path": p, "filename": f"{c}/{p.name}", "label": c}
            for c in CLASSES for p in sorted((split_dir / c).iterdir()) if p.suffix.lower() in IMG_EXTS]
    return pd.DataFrame(rows)

train_df, test_df = list_images(TRAIN_DIR), list_images(TEST_DIR)

counts = pd.DataFrame({"train": train_df["label"].value_counts(),
                       "test": test_df["label"].value_counts()}).reindex(CLASSES)
counts.loc["TOTAL"] = counts.sum()
counts["total"] = counts["train"] + counts["test"]
counts["train %"] = (100 * counts["train"] / counts["total"]).round(1)
counts.index.name = "emotion"
print("\nImages per class")
print(counts.to_string())
counts.to_csv(OUT_DIR / "split_counts.csv")

In [ ]:
# Image spec: every file should decode as a 48x48 single-channel image.
# The same pass hashes the pixels so we can check for identical images across the two splits.
def pixel_hash(img):
    return hashlib.md5(img.tobytes()).hexdigest()

specs, hashes = {}, {}
for split, df in (("train", train_df), ("test", test_df)):
    shapes, hs = [], []
    for p in df["path"]:
        img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
        assert img is not None, f"Could not read {p}"
        shapes.append(img.shape)
        hs.append(pixel_hash(img))
    specs[split] = pd.Series(shapes).value_counts()
    hashes[split] = set(hs)
    df["hash"] = hs

for split, s in specs.items():
    print(f"{split}: image shapes ->", {str(k): int(v) for k, v in s.items()})
cross_dups = hashes["train"] & hashes["test"]
print(f"\nTrain/test filename overlap: {len(set(train_df['path'].map(lambda p: p.name)) & set(test_df['path'].map(lambda p: p.name)))}")
print(f"Pixel-identical images found in BOTH train and test: {len(cross_dups)} "
      f"({100 * test_df['hash'].isin(cross_dups).mean():.2f}% of test images) - a known quirk of FER2013")

In [ ]:
ax = counts.drop("TOTAL")[["train", "test"]].plot(kind="bar", figsize=(9, 4), rot=0, color=["#4C78A8", "#F58518"])
ax.set_title("FER2013 class distribution (train vs test)")
ax.set_xlabel("Emotion")
ax.set_ylabel("Number of images")
for cont in ax.containers:
    ax.bar_label(cont, fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "class_distribution.png", dpi=150)
plt.show()

In [ ]:
# A few raw samples per class, to confirm the images display correctly
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(9, 11))
for r, c in enumerate(CLASSES):
    paths = train_df.loc[train_df["label"] == c, "path"].to_numpy()
    for k, p in enumerate(rng.choice(paths, 6, replace=False)):
        axes[r, k].imshow(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE), cmap="gray")
        axes[r, k].axis("off")
    axes[r, 0].set_title(c, loc="left", fontsize=11, fontweight="bold")
plt.suptitle("FER2013 training samples (6 per class)")
plt.tight_layout()
plt.savefig(OUT_DIR / "sample_grid.png", dpi=150)
plt.show()

In [ ]:
# Hold out one test image per class as "unseen" images for the predict-function demo.
# They are removed from the evaluation set so the demo images are not used anywhere else.
holdout = test_df.groupby("label", sort=False).sample(1, random_state=SEED).reset_index(drop=True)
unseen_rows = []
for i, row in holdout.iterrows():
    dst = UNSEEN_DIR / f"unseen_{i + 1:02d}.jpg"
    shutil.copy(row["path"], dst)
    unseen_rows.append({"file": dst.name, "source": row["filename"], "true_label": row["label"]})
unseen_truth = pd.DataFrame(unseen_rows)
unseen_truth.to_csv(UNSEEN_DIR / "unseen_labels.csv", index=False)

test_df = test_df[~test_df["path"].isin(holdout["path"])].reset_index(drop=True)
print(f"Held out {len(holdout)} images to {UNSEEN_DIR.name}/ ; evaluation test set is now {len(test_df)} images")
print(unseen_truth.to_string(index=False))

## STEP 2 — Milestone 1: Face detection & visual check
OpenCV's Haar cascade (`haarcascade_frontalface_default.xml`) is used as the face detector. FER2013 faces are tiny (48×48) and already tightly cropped, and the cascade's sliding window is 24×24, so each image is first padded with a 12-px replicated border (the cascade needs a little context around the face; this raised the detection rate from about 66% to about 82% in a quick test), upscaled ×4 and histogram-equalized. Boxes are mapped back to, and clipped to, the original 48×48 frame. When several faces are found, the largest box is used.

In [ ]:
face_cascade = cv2.CascadeClassifier(str(HAAR_PATH))
assert not face_cascade.empty(), f"Could not load {HAAR_PATH}"

def detect_faces(gray):
    # Return face boxes (x, y, w, h) in the coordinates of `gray`, largest first
    H, W = gray.shape[:2]
    small = max(H, W) < 200                           # FER-sized input: pad + upscale
    pad, scale = (H // 4, 4) if small else (0, 1)
    img = cv2.copyMakeBorder(gray, pad, pad, pad, pad, cv2.BORDER_REPLICATE) if pad else gray
    img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale > 1 else img
    img = cv2.equalizeHist(img)
    min_side = max(24, min(img.shape) // 4)
    raw = face_cascade.detectMultiScale(img, scaleFactor=1.05, minNeighbors=4, minSize=(min_side, min_side))
    boxes = []
    for x, y, w, h in raw:                            # map back to the original frame and clip to it
        x0, y0 = max(0, round(x / scale) - pad), max(0, round(y / scale) - pad)
        x1, y1 = min(W, round((x + w) / scale) - pad), min(H, round((y + h) / scale) - pad)
        if x1 - x0 > 4 and y1 - y0 > 4:
            boxes.append((x0, y0, x1 - x0, y1 - y0))
    return sorted(boxes, key=lambda b: b[2] * b[3], reverse=True)

def crop_face(gray, margin=0.10):
    # Crop the largest detected face (+10% margin). FER2013 images are already face crops,
    # so if the detector finds nothing we fall back to the full frame.
    boxes = detect_faces(gray)
    if not boxes:
        return gray, None
    x, y, w, h = boxes[0]
    mx, my = int(w * margin), int(h * margin)
    x0, y0 = max(0, x - mx), max(0, y - my)
    x1, y1 = min(gray.shape[1], x + w + mx), min(gray.shape[0], y + h + my)
    return gray[y0:y1, x0:x1], (x, y, w, h)

In [ ]:
# Draw boxes on 4 random training samples per class
rng = np.random.default_rng(SEED + 1)
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(8, 14))
for r, c in enumerate(CLASSES):
    paths = train_df.loc[train_df["label"] == c, "path"].to_numpy()
    for k, p in enumerate(rng.choice(paths, 4, replace=False)):
        gray = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        vis = cv2.cvtColor(cv2.resize(gray, (192, 192), interpolation=cv2.INTER_CUBIC), cv2.COLOR_GRAY2RGB)
        boxes = detect_faces(gray)
        for j, (x, y, w, h) in enumerate(boxes):
            color = (0, 255, 0) if j == 0 else (255, 160, 0)   # green = largest face (used), orange = others
            cv2.rectangle(vis, (4 * x, 4 * y), (4 * (x + w), 4 * (y + h)), color, 2)
        axes[r, k].imshow(vis)
        axes[r, k].set_title(f"{c}: {len(boxes)} face(s)" if boxes else f"{c}: none -> full frame",
                             fontsize=9, color="black" if boxes else "red")
        axes[r, k].axis("off")
plt.suptitle("Milestone 1 - Haar face detection on FER2013 (green = box used for cropping)")
plt.tight_layout()
plt.savefig(OUT_DIR / "face_detection.png", dpi=150)
plt.show()

In [ ]:
# Detection rate on a random sample of 200 training images per class (the full-dataset rate is reported in STEP 3)
rng = np.random.default_rng(SEED + 2)
rows = []
for c in CLASSES:
    paths = train_df.loc[train_df["label"] == c, "path"].to_numpy()
    sample = rng.choice(paths, min(200, len(paths)), replace=False)
    found = [len(detect_faces(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE))) > 0 for p in sample]
    rows.append({"emotion": c, "sampled": len(sample), "face found": int(np.sum(found)),
                 "detection rate %": round(100 * np.mean(found), 1)})
det_sample = pd.DataFrame(rows).set_index("emotion")
print(det_sample.to_string())

## STEP 3 — Milestone 2 (Track B): CNN deep features with MobileNetV2
For each image: crop the largest detected face → resize to 224×224 → copy the grayscale channel into 3 channels → ImageNet mean/std normalization → pretrained **MobileNetV2** (ImageNet weights, frozen) → global average pooling of the last convolutional block → a **1280-dimensional embedding**. Embeddings are cached in `features/` so the notebook can be re-run quickly.

In [ ]:
WEIGHTS = MobileNet_V2_Weights.IMAGENET1K_V2
backbone = mobilenet_v2(weights=WEIGHTS).eval()      # frozen feature extractor, no fine-tuning
IMG_SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

def preprocess_face(gray):
    # gray (H, W) uint8 -> (3, 224, 224) float32 tensor input, plus whether a face was detected
    face, box = crop_face(gray)
    face = cv2.resize(face, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    rgb = np.repeat(face[..., None], 3, axis=2).astype(np.float32) / 255.0
    rgb = (rgb - MEAN) / STD
    return rgb.transpose(2, 0, 1), box is not None

@torch.inference_mode()
def embed(batch):
    x = torch.from_numpy(np.stack(batch))
    fmap = backbone.features(x)
    return torch.nn.functional.adaptive_avg_pool2d(fmap, 1).flatten(1).numpy()

def extract_features(df, cache_path, batch_size=128):
    if cache_path.exists():
        z = np.load(cache_path, allow_pickle=False)
        if list(z["filename"]) == list(df["filename"]):
            print(f"Loaded cached features from {cache_path.name}")
            return z["X"], z["face_found"]
    feats, found, batch = [], [], []
    t0 = time.time()
    for i, p in enumerate(df["path"], 1):
        x, ok = preprocess_face(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE))
        batch.append(x)
        found.append(ok)
        if len(batch) == batch_size or i == len(df):
            feats.append(embed(batch))
            batch = []
        if i % 5000 == 0 or i == len(df):
            print(f"  {i:>6}/{len(df)} images  ({time.time() - t0:.0f}s)")
    X, found = np.concatenate(feats).astype(np.float32), np.array(found)
    np.savez_compressed(cache_path, X=X, face_found=found, filename=df["filename"].to_numpy(dtype=str))
    return X, found

print("Train embeddings:")
X_train, found_train = extract_features(train_df, FEAT_DIR / "train_mobilenetv2.npz")
print("Test embeddings:")
X_test, found_test = extract_features(test_df, FEAT_DIR / "test_mobilenetv2.npz")
y_train, y_test = train_df["label"].to_numpy(), test_df["label"].to_numpy()

In [ ]:
# Train-test split verification (shapes, label coverage, detection rate on the full dataset)
print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")
print(f"Train/test ratio: {len(y_train) / (len(y_train) + len(y_test)):.1%} / {len(y_test) / (len(y_train) + len(y_test)):.1%}")
print("Classes in train:", sorted(set(y_train)) == sorted(CLASSES), "| classes in test:", sorted(set(y_test)) == sorted(CLASSES))
print("Any NaN/inf in features:", bool(~np.isfinite(X_train).all() or ~np.isfinite(X_test).all()))

det_full = pd.DataFrame({
    "train face found %": pd.Series(found_train).groupby(y_train).mean().mul(100).round(1),
    "test face found %": pd.Series(found_test).groupby(y_test).mean().mul(100).round(1),
}).reindex(CLASSES)
det_full.loc["ALL"] = [round(100 * found_train.mean(), 1), round(100 * found_test.mean(), 1)]
print("\nHaar detection rate (rest fall back to the full 48x48 frame):")
print(det_full.to_string())
det_full.to_csv(OUT_DIR / "detection_rate.csv")

## STEP 4 — Model training (two classical classifiers)
* **Logistic Regression**: `StandardScaler` → multinomial logistic regression (L2, lbfgs). The regularization strength `C` is picked on a 10% validation split carved out of the *training* set (the test set is not used for tuning). `class_weight="balanced"` compensates for the very small *disgust* class.
* **Random Forest**: 300 trees, `max_features="sqrt"`, `min_samples_leaf=2`, `class_weight="balanced_subsample"`. Trees are scale-invariant, so no standardization.

Both use `random_state=42`.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.10, stratify=y_train, random_state=SEED)

def make_logreg(C):
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(C=C, max_iter=3000, class_weight="balanced", random_state=SEED))])

val_rows = []
for C in (0.001, 0.01, 0.1, 1.0):
    t0 = time.time()
    m = make_logreg(C).fit(X_tr, y_tr)
    pred = m.predict(X_val)
    val_rows.append({"C": C, "val accuracy": accuracy_score(y_val, pred),
                     "val macro-F1": f1_score(y_val, pred, average="macro"), "fit time (s)": time.time() - t0})
val_table = pd.DataFrame(val_rows)
print("Logistic Regression - validation search over C")
print(val_table.round(4).to_string(index=False))
BEST_C = float(val_table.loc[val_table["val macro-F1"].idxmax(), "C"])
print(f"\nSelected C = {BEST_C}")

In [ ]:
models = {
    "Logistic Regression": make_logreg(BEST_C),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_features="sqrt", min_samples_leaf=2,
                                            class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)          # refit on the full training set
    print(f"{name:<20} trained on {X_train.shape[0]} x {X_train.shape[1]} features in {time.time() - t0:.1f}s")
    print("   ", model)

## STEP 5 — Milestone 3: Evaluation

In [ ]:
results, predictions, probas = [], {}, {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name], probas[name] = y_pred, model.predict_proba(X_test)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=CLASSES, average="macro", zero_division=0)
    results.append({"Model": name, "Accuracy": accuracy_score(y_test, y_pred),
                    "Macro-Precision": p, "Macro-Recall": r, "Macro-F1": f1})
results_df = pd.DataFrame(results).set_index("Model")
results_df.round(4).to_csv(OUT_DIR / "model_comparison.csv")
print(f"Test set: {len(y_test)} images\n")
print(results_df.round(4).to_string())

In [ ]:
for name, y_pred in predictions.items():
    report = classification_report(y_test, y_pred, labels=CLASSES, digits=4, zero_division=0)
    (OUT_DIR / f"classification_report_{name.replace(' ', '_').lower()}.txt").write_text(report)
    print(f"=== {name} ===")
    print(report)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for col, (name, y_pred) in enumerate(predictions.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=CLASSES, ax=axes[0, col], cmap="Blues",
                                            colorbar=False, xticks_rotation=45)
    axes[0, col].set_title(f"{name} - counts\nAccuracy = {accuracy_score(y_test, y_pred):.3f}")
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=CLASSES, ax=axes[1, col], cmap="Blues",
                                            colorbar=False, xticks_rotation=45, normalize="true", values_format=".2f")
    axes[1, col].set_title(f"{name} - row-normalized (recall per true class)")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
# Most common confusions (share of each true class predicted as another class)
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=CLASSES, normalize="true")
    np.fill_diagonal(cm, 0)
    top = sorted(((cm[i, j], CLASSES[i], CLASSES[j]) for i in range(7) for j in range(7)), reverse=True)[:5]
    print(f"{name}: top confusions (true -> predicted)")
    for v, t, p in top:
        print(f"   {t:>8} -> {p:<8} {v:.1%} of true {t}")

In [ ]:
ax = results_df.plot(kind="bar", figsize=(9, 4), rot=0, color=["#4C78A8", "#F58518", "#54A24B", "#E45756"])
ax.set_ylim(0, 1)
ax.set_title("Model comparison on the FER2013 test set")
ax.set_ylabel("Score")
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.3f", fontsize=8)
ax.legend(loc="upper right", ncol=4, fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "model_comparison.png", dpi=150)
plt.show()

In [ ]:
# Misclassified examples from the better model (highest-confidence mistakes)
best_name = results_df["Macro-F1"].idxmax()
y_pred, proba = predictions[best_name], probas[best_name]
conf = proba.max(axis=1)
wrong = np.where(y_pred != y_test)[0]
wrong = wrong[np.argsort(-conf[wrong])][:8]
fig, axes = plt.subplots(2, 4, figsize=(11, 6))
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(cv2.imread(str(test_df.loc[i, "path"]), cv2.IMREAD_GRAYSCALE), cmap="gray")
    ax.set_title(f"{test_df.loc[i, 'filename'].split('/')[1]}\ntrue: {y_test[i]} | pred: {y_pred[i]} ({conf[i]:.2f})", fontsize=8)
    ax.axis("off")
plt.suptitle(f"Most confident mistakes - {best_name}")
plt.tight_layout()
plt.savefig(OUT_DIR / "misclassified_examples.png", dpi=150)
plt.show()

## STEP 6 — Simple deployment: `predict_emotion()`
The better model (highest macro-F1) is wrapped in one function that takes an image path, runs the **same** preprocessing and feature extraction (Haar crop → 224×224 → 3 channels → MobileNetV2 embedding) and returns one of the seven emotion labels. It works on FER-style 48×48 images and on larger photos (any image OpenCV can read; the largest face is used).

In [ ]:
best_model = models[best_name]

def predict_emotion(image_path, model=best_model):
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    x, face_found = preprocess_face(gray)
    proba = model.predict_proba(embed([x]))[0]
    order = np.argsort(-proba)
    return {"label": model.classes_[order[0]], "confidence": float(proba[order[0]]), "face_found": face_found,
            "top3": ", ".join(f"{model.classes_[k]} {proba[k]:.2f}" for k in order[:3])}

print(f"Deployed model: {best_name}")
print(predict_emotion(UNSEEN_DIR / "unseen_01.jpg"))

In [ ]:
# Predictions on every image in unseen_images/ (held out above; never used for training or evaluation)
truth = dict(zip(unseen_truth["file"], unseen_truth["true_label"]))
paths = sorted(p for p in UNSEEN_DIR.iterdir() if p.suffix.lower() in IMG_EXTS)
rows = []
for p in paths:
    out = predict_emotion(p)
    rows.append({"filename": p.name, "predicted": out["label"], "confidence": round(out["confidence"], 3),
                 "face found": out["face_found"], "top-3": out["top3"], "true label": truth.get(p.name, "?")})
pred_df = pd.DataFrame(rows)
pred_df.to_csv(OUT_DIR / "unseen_predictions.csv", index=False)
print(pred_df.to_string(index=False))
known = pred_df[pred_df["true label"] != "?"]
print(f"\nCorrect: {(known['predicted'] == known['true label']).sum()} / {len(known)}")

fig, axes = plt.subplots(1, len(paths), figsize=(2.3 * len(paths), 3.2))
for ax, p, (_, r) in zip(np.atleast_1d(axes), paths, pred_df.iterrows()):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ok = r["predicted"] == r["true label"]
    ax.set_title(f"{r['filename']}\npred: {r['predicted']} ({r['confidence']:.2f})\ntrue: {r['true label']}",
                 fontsize=8, color="green" if ok else "red")
    ax.axis("off")
plt.suptitle(f"predict_emotion() on unseen images - {best_name}")
plt.tight_layout()
plt.savefig(OUT_DIR / "unseen_predictions.png", dpi=150)
plt.show()